#Preprocessing + Feature Engineering

Two pipelines:
- **Pipeline A**: Random Forest + XGBoost
- **Pipeline B**: CNN-BiGRU

load `combined_raw.csv` because we need the original URL for feature extraction

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/kwago/'

Mounted at /content/drive


In [ ]:
!pip install nltk scikit-learn imbalanced-learn -q
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
!pip install skl2onnx -q
!pip install skl2onnx onnxmltools onnxruntime onnx -q
import pandas as pd
import numpy as np
import re
import pickle
import scipy.sparse as sp
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType, FloatTensorType
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from imblearn.over_sampling import SMOTE


df = pd.read_csv(BASE + 'combined_raw.csv')
print(f'Loaded: {df.shape}')
print(df['Classification'].value_counts())
df[['Message','Classification']].head(3)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.0/304.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.3 MB/s eta 0:00:00
Loaded: (13535, 6)
Classification
Benign      10134
Phishing     3401
Name: count, dtype: int64


,Message,Classification
0,Never share your OTP 385535 or enter it on sus...,Benign
1,Secure your accounts and cards your way with B...,Benign
2,This deal's as hot as summer. Get GOMO FIBER a...,Benign


## Feature

In [ ]:
PH_BANKS = [
    'bdo','bpi','metrobank','landbank','rcbc','unionbank',
    'eastwest','psbank','chinabank','security bank','pnb',
    'gcash','maya','paymaya','gotyme','seabank','tonik',
]
PH_TELCOS = ['smart','globe','tnt','sun','dito','gomo','tm']
PH_URGENCY = [
    'agad','ngayon','mawala','deadline','huling araw','expir',
    'panalo','manalo','libreng','libre','premyo','reward',
    'kunin','i-click','i-verify','i-update','i-confirm',
    'mag-claim','i-redeem','i-activate','mag-log','mag-login',
]
URL_SHORTENERS = [
    'bit.ly','tinyurl','t.co','goo.gl','ow.ly',
    'short.link','rb.gy','cutt.ly','tiny.cc','is.gd',
]
CTA_PHRASES = [
    'click here','verify now','claim your','act now','limited time',
    'expires today','call now','text now','reply now','visit now',
    'click link','tap here','open now','log in now','sign in now',
    'update now','confirm now','validate now','redeem now',
]

def extract_urls(text):
    return re.findall(r'https?://\S+|www\.\S+', str(text))

def get_url_features(text):
    urls = extract_urls(text)
    if not urls:
        return {'url_present':0,'url_count':0,'has_shortener':0,
                'has_https':0,'domain_length':0,'subdomain_count':0,
                'has_ip':0,'path_depth':0,'url_special_chars':0}
    url = urls[0]
    domain_match = re.search(r'https?://([^/]+)', url)
    domain = domain_match.group(1) if domain_match else ''
    path = re.sub(r'https?://[^/]+','',url)
    return {
        'url_present': 1,
        'url_count': len(urls),
        'has_shortener': int(any(s in url.lower() for s in URL_SHORTENERS)),
        'has_https': int(url.startswith('https')),
        'domain_length': len(domain),
        'subdomain_count': max(0, domain.count('.')-1),
        'has_ip': int(bool(re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', domain))),
        'path_depth': len([p for p in path.split('/') if p]),
        'url_special_chars': len(re.findall(r'[-_~%@]', url)),
    }

def get_structural_features(text):
    t = str(text)
    return {
        'char_count': len(t),
        'word_count': len(t.split()),
        'punct_count': len(re.findall(r'[.,!?]', t)),
        'digit_density': round(sum(c.isdigit() for c in t)/len(t),4) if t else 0,
        'upper_ratio': round(sum(c.isupper() for c in t)/len(t),4) if t else 0,
    }

def get_behavioral_features(text):
    t = str(text).lower()
    return {
        'has_cta': int(any(p in t for p in CTA_PHRASES)),
        'cta_count': sum(t.count(p) for p in CTA_PHRASES),
    }

def get_ph_context_features(text):
    t = str(text).lower()
    return {
        'has_ph_bank': int(any(b in t for b in PH_BANKS)),
        'has_ph_telco': int(any(te in t for te in PH_TELCOS)),
        'has_ph_urgency': int(any(u in t for u in PH_URGENCY)),
    }

print('Extracting features...')
url_feats  = df['Message'].apply(get_url_features).apply(pd.Series)
struct_feats = df['Message'].apply(get_structural_features).apply(pd.Series)
beh_feats  = df['Message'].apply(get_behavioral_features).apply(pd.Series)
ph_feats   = df['Message'].apply(get_ph_context_features).apply(pd.Series)

df_features = pd.concat([url_feats, struct_feats, beh_feats, ph_feats], axis=1)
print(f'Feature matrix shape: {df_features.shape}')
print(f'Features: {df_features.columns.tolist()}')
print()
df_features.head(3)

Extracting features...
Feature matrix shape: (13535, 19)
Features: ['url_present', 'url_count', 'has_shortener', 'has_https', 'domain_length', 'subdomain_count', 'has_ip', 'path_depth', 'url_special_chars', 'char_count', 'word_count', 'punct_count', 'digit_density', 'upper_ratio', 'has_cta', 'cta_count', 'has_ph_bank', 'has_ph_telco', 'has_ph_urgency']



,url_present,url_count,has_shortener,has_https,domain_length,subdomain_count,has_ip,path_depth,url_special_chars,char_count,word_count,punct_count,digit_density,upper_ratio,has_cta,cta_count,has_ph_bank,has_ph_telco,has_ph_urgency
0,0,0,0,0,0,0,0,0,0,157.0,29.0,4.0,0.0764,0.0764,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,279.0,46.0,7.0,0.0036,0.0860,1,1,1,0,0
2,0,0,0,0,0,0,0,0,0,258.0,50.0,7.0,0.0698,0.1318,0,0,0,1,0


## Clean Text (after feature extraction)

Now we clean the message text. URLs have already been captured as features so it's safe to remove them now.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

def clean_text(text: str) -> str:
    if not isinstance(text, str): return ''
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'\+?63[\d\*]{9,10}', '', text)
    text = re.sub(r'\b0\d{10}\b', '', text)
    text = re.sub(r'\b\d{3,4}[-\s]?\d{3,4}[-\s]?\d{4}\b', '', text)
    emoji_pattern = re.compile(
        u'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF'
        u'\U00002702-\U000027B0\U000024C2-\U0001F251]+',
        flags=re.UNICODE)
    text = emoji_pattern.sub('', text)
    text = re.sub(r"[^a-zA-Z0-9\u00C0-\u024F\s.,!?'\-]", '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Cleaned_Message'] = df['Message'].apply(clean_text)

before = len(df)
df = df[df['Cleaned_Message'].str.strip() != ''].reset_index(drop=True)
df_features = df_features.loc[df.index].reset_index(drop=True)
print(f'Dropped empty after cleaning : {before - len(df)} rows')

before = len(df)
df = df.drop_duplicates(subset=['Cleaned_Message','Classification']).reset_index(drop=True)
df_features = df_features.loc[df.index].reset_index(drop=True)
print(f'Dropped duplicates : {before - len(df)} rows')
print(f'Remaining : {len(df):,} rows')

Dropped empty after cleaning : 11 rows
Dropped duplicates : 2614 rows
Remaining : 10,910 rows


## Train / Validation / Test Split (70/15/15)

We split first, then fit TF-IDF and MinMax scaler on training data only. This prevents data leakage — the model should never see validation or test data during training.

##RF + XGBoost

In [ ]:
X_text = df['Cleaned_Message']
y = df['Classification']
X_feat = df_features

X_text_train, X_text_temp, X_feat_train, X_feat_temp, y_train, y_temp = train_test_split(
    X_text, X_feat, y, test_size=0.30, random_state=42, stratify=y)

X_text_val,  X_text_test,  X_feat_val,  X_feat_test,  y_val, y_test = train_test_split(
    X_text_temp, X_feat_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train : {len(X_text_train):,} rows ({len(X_text_train)/len(X_text)*100:.1f}%)')
print(f'Validation : {len(X_text_val):,} rows ({len(X_text_val)/len(X_text)*100:.1f}%)')
print(f'Test : {len(X_text_test):,} rows ({len(X_text_test)/len(X_text)*100:.1f}%)')
print()
print('Train class distribution:')
print(y_train.value_counts())

## CNN-BiGRU

In [ ]:
from nltk.corpus import stopwords

stop_tl = {
    'ang','ng','na','sa','at','ay','ko','ka','mo','niya','namin','natin',
    'nila','siya','kami','tayo','sila','mga','hindi','po','opo','yung',
    'nang','para','may','wala','ito','iyon','dito','doon','kaya','kung',
    'pero','dahil','kasi','lang','rin','din','pa','naman','talaga','ba',
    'nga','pala','raw','daw','ho','ano','sino','saan','bakit','alin',
}
all_stop = set(stopwords.words('english')) | stop_tl

PROTECTED_TOKENS = {'httpurltoken', 'phonenumtoken'}

def preprocess_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip(): return ''
    text = re.sub(r'(?<!\w)\d+(?!\w)', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split()
              if (w in PROTECTED_TOKENS or w not in all_stop) and len(w) > 1]
    return ' '.join(tokens)

X_text_train_a = X_text_train.apply(preprocess_text)
X_text_val_a   = X_text_val.apply(preprocess_text)
X_text_test_a  = X_text_test.apply(preprocess_text)


tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)
X_tfidf_train = tfidf.fit_transform(X_text_train_a)
X_tfidf_val   = tfidf.transform(X_text_val_a)
X_tfidf_test  = tfidf.transform(X_text_test_a)
print(f'TF-IDF shape (train) : {X_tfidf_train.shape}')


scaler = MinMaxScaler()
X_struct_train = scaler.fit_transform(X_feat_train)
X_struct_val   = scaler.transform(X_feat_val)
X_struct_test  = scaler.transform(X_feat_test)
print(f'Structured features  : {X_struct_train.shape}')


X_train_combined = hstack([X_tfidf_train, csr_matrix(X_struct_train)])
X_val_combined   = hstack([X_tfidf_val,   csr_matrix(X_struct_val)])
X_test_combined  = hstack([X_tfidf_test,  csr_matrix(X_struct_test)])
print(f'Combined shape (train): {X_train_combined.shape}')
print(f'Combined shape (val)  : {X_val_combined.shape}')
print(f'Combined shape (test) : {X_test_combined.shape}')

print('\nBefore SMOTE:')
print(pd.Series(y_train).value_counts())
smote_a = SMOTE(random_state=42)
X_train_A, y_train_A = smote_a.fit_resample(X_train_combined, y_train)
print('\nAfter SMOTE:')
print(pd.Series(y_train_A).value_counts())

## Tokenization

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

MAX_WORDS = 10000
MAX_LEN = 150

tokenizer_b = Tokenizer(num_words=MAX_WORDS, oov_token='<UNK>')
tokenizer_b.fit_on_texts(X_text_train)

X_train_seq = tokenizer_b.texts_to_sequences(X_text_train)
X_val_seq   = tokenizer_b.texts_to_sequences(X_text_val)
X_test_seq  = tokenizer_b.texts_to_sequences(X_text_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad   = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

## Save Preprocessing Objects

We save the TF-IDF vectorizer, scaler, and tokenizer so they can be loaded in the modelling notebook without refitting.

In [ ]:
with open(BASE + 'tfidf.pkl', 'wb') as f:   pickle.dump(tfidf, f)
with open(BASE + 'scaler.pkl', 'wb') as f:  pickle.dump(scaler, f)
with open(BASE + 'tokenizer_b.pkl', 'wb') as f:    pickle.dump(tokenizer_b, f)
with open(BASE + 'label_encoder.pkl', 'wb') as f:  pickle.dump(le, f)

n_struct_features = X_feat_train.shape[1]
scaler_initial_type = [('num_input', FloatTensorType([None, n_struct_features]))]
tfidf_initial_type = [('text_input', StringTensorType([None, 1]))]

scaler_onnx = convert_sklearn(scaler, initial_types=scaler_initial_type)
tfidf_onnx = convert_sklearn(tfidf, initial_types=tfidf_initial_type)

with open(BASE + 'scaler.onnx', 'wb') as f: f.write(scaler_onnx.SerializeToString())
with open(BASE + 'tfidf.onnx', 'wb') as f: f.write(tfidf_onnx.SerializeToString())

sp.save_npz(BASE + 'X_train_A.npz', X_train_A)
sp.save_npz(BASE + 'X_val_A.npz',   X_val_combined)
sp.save_npz(BASE + 'X_test_A.npz',  X_test_combined)

if y_train_A.dtype == object:
    y_train_A_enc = le.transform(y_train_A)
else:
    y_train_A_enc = y_train_A.astype(int)

np.save(BASE + 'y_train_A.npy', y_train_A_enc)
np.save(BASE + 'X_train_B.npy', X_train_pad)
np.save(BASE + 'X_val_B.npy',   X_val_pad)
np.save(BASE + 'X_test_B.npy',  X_test_pad)
np.save(BASE + 'y_train_B.npy', y_train_enc)
np.save(BASE + 'y_val.npy',     y_val_enc)
np.save(BASE + 'y_test.npy',    y_test_enc)